In [10]:
from dotenv import load_dotenv
from os import getenv
import vk_utils
from postgres_utils import connect_pgsql


import importlib
importlib.reload(vk_utils)


load_dotenv()

True

In [11]:
with connect_pgsql(getenv("POSTGRESQL_DSN"), timeout_seconds=10) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "CREATE TABLE IF NOT EXISTS vk_monitor_polling_list ("
            "group_id BIGINT PRIMARY KEY, "
            "last_read_post_id BIGINT"
            ")"
        )
        cur.execute(
            "INSERT INTO vk_monitor_polling_list (group_id) "
            "VALUES (%s) ON CONFLICT (group_id) DO NOTHING",
            (237677627,),
        )

In [12]:
new_posts = vk_utils.poll_vk_new_messages(
    api_key=getenv("VK_API_KEY"),
    state_dsn=getenv("POSTGRESQL_DSN")
)
new_posts

[{'group_id': 237677627,
  'post_id': 20,
  'text': 'Тест 9. Улица Образцова, 9',
  'latitude': None,
  'longitude': None},
 {'group_id': 237677627,
  'post_id': 19,
  'text': 'Тест 8',
  'latitude': '55.788197571617',
  'longitude': '37.606734691399'}]

In [13]:
import geolocation_utils

import importlib
importlib.reload(geolocation_utils)


new_posts_with_geo = geolocation_utils.enrich_posts_with_coords(posts=new_posts, openai_api=getenv(
    "OPENAI_API"), openai_api_key=getenv("OPENAI_API_KEY"), openai_model=getenv("OPENAI_API_MODEL"))
new_posts_with_geo

[{'group_id': 237677627,
  'post_id': 20,
  'text': 'Тест 9. Улица Образцова, 9',
  'latitude': 55.788536,
  'longitude': 37.6088108},
 {'group_id': 237677627,
  'post_id': 19,
  'text': 'Тест 8',
  'latitude': '55.788197571617',
  'longitude': '37.606734691399'}]